# 1. One-Hot Encoding

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import matthews_corrcoef
import numpy as np
import pandas as pd


In [2]:
def create_one_hot_sets(dataset):
    """
    Processes the dataset by splitting it into predefined sets,
    standardizing sequence lengths, and applying One-Hot Encoding.
    """
    sets = ["1", "2", "3", "4", "5"]
    set_results = []
    for i in sets:
        st = dataset.query(f"Set == '{i}'")
        tmp_x, tmp_y = [], []
        for _, row in st.iterrows():
            seq = row["Sequence"]
            seq = _increase_lenseq(seq) if len(seq) < 90 else seq[:90]
            tmp_x.append(one_hot_encoding(seq))
            tmp_y.append(1 if row["Class"] == "Positive" else 0)
        set_results.append((
            np.array(tmp_x, dtype=np.float32),
            np.array(tmp_y, dtype=np.float32)
        ))
    return set_results


def _increase_lenseq(seq):
    """Pads the sequence with 'X' characters to reach length 90."""
    return seq + "X" * (90 - len(seq))


def one_hot_encoding(sequence):
    """
    Transforms an amino acid sequence into a binary matrix.
    Output shape: (Sequence_Length, 21)
    """
    aa_alph = ['A','C','D','E','F','G','H','I','K','L',
               'M','N','P','Q','R','S','T','V','W','Y','X']
    M = []
    for aa in sequence:
        one_hot = np.zeros(21)
        try:
            one_hot[aa_alph.index(aa)] = 1
        except ValueError:
            pass
        M.append(one_hot)
    return np.array(M)


# 2. Network Definition and Infrastructure

In [3]:
class SP_NN(nn.Module):
    def __init__(self, input_size, hidden_sizes, lstm_hidden_size,
                 num_lstm_layers, output_size, dropout_p=0.5):
        """
        Hybrid CNN-LSTM model for Signal Peptide prediction.

        Args:
            input_size (int): Dimension of the input vector (21 for One-Hot).
            hidden_sizes (list): List of integers for the MLP head architecture.
            lstm_hidden_size (int): Number of features in the LSTM hidden state.
            num_lstm_layers (int): Number of stacked LSTM layers.
            output_size (int): Size of the output (1 for binary classification).
            dropout_p (float): Dropout probability for regularization.
        """
        super(SP_NN, self).__init__()

        # 1. Convolutional Block
        self.cnn_out_channels = 64
        self.conv1 = nn.Conv1d(input_size, self.cnn_out_channels,
                               kernel_size=17, padding='same')

        # 2. Recurrent Block (LSTM)
        self.lstm = nn.LSTM(
            self.cnn_out_channels, lstm_hidden_size, num_lstm_layers,
            batch_first=True,
            dropout=dropout_p if num_lstm_layers > 1 else 0
        )
        self.bn = nn.BatchNorm1d(lstm_hidden_size)

        # 3. Dynamic MLP Head
        mlp_layers = []
        current_input_size = lstm_hidden_size
        for hidden_size in hidden_sizes:
            mlp_layers.append(nn.Linear(current_input_size, hidden_size))
            mlp_layers.append(nn.ReLU())
            mlp_layers.append(nn.Dropout(p=dropout_p))
            current_input_size = hidden_size
        mlp_layers.append(nn.Linear(current_input_size, output_size))
        mlp_layers.append(nn.Sigmoid())
        self.mlp = nn.Sequential(*mlp_layers)

    def forward(self, x):
        # x: [Batch, Length=90, Channels=21]
        x = x.permute(0, 2, 1)          # -> [Batch, 21, 90]
        x = self.conv1(x)               # -> [Batch, 64, 90]
        x = x.permute(0, 2, 1)          # -> [Batch, 90, 64]
        out, _ = self.lstm(x)           # -> [Batch, 90, lstm_hidden]
        out = out[:, -1, :]             # -> [Batch, lstm_hidden]
        out = self.bn(out)
        out = self.mlp(out)
        return out


class SignalDataset(Dataset):
    """Lazy-loading Dataset: converts data to Tensors only on demand."""

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x_out = torch.from_numpy(self.X[idx]).float()
        y_out = torch.tensor(self.y[idx], dtype=torch.float32).view(1)
        return x_out, y_out


# FIX 1 & 2: train_val() — corrected indentation throughout the function
# and moved the docstring inside the function body.
def train_val(model, train_loader, val_loader, optimizer, criterion,
              epochs, patience,
              scorer=matthews_corrcoef,
              init_best_score=-1,
              output_transform=lambda x: (x > 0.5).float()):
    """
    Training and validation loop with Early Stopping and Gradient Clipping.

    Returns:
        dict: state_dict of the best performing model found during training.
    """
    best_val_score = init_best_score
    epochs_without_improvement = 0
    best_model_state_dict = None

    for epoch in range(epochs):
        # ── Training Phase ──────────────────────────────────────────
        model.train()
        loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        # ── Validation Phase ─────────────────────────────────────────
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                preds = output_transform(outputs)
                val_preds.extend(preds.cpu().numpy().flatten())
                val_labels.extend(batch_y.cpu().numpy().flatten())

        val_score = scorer(val_labels, val_preds)

        # ── Early Stopping ───────────────────────────────────────────
        if val_score > best_val_score:
            best_val_score = val_score
            epochs_without_improvement = 0
            best_model_state_dict = model.state_dict()
            print(f'Validation score improved to {best_val_score:.4f}')
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'Early stopping at epoch {epoch + 1}')
                break

        print(f'Epoch [{epoch+1}/{epochs}], '
              f'Loss: {loss.item():.4f}, Val MCC: {val_score:.4f}')

    return best_model_state_dict


# FIX 3: test() — fixed indentation (model.eval() and score= were at
# 2-space / 1-space indent, i.e. outside the function body).
# Also now returns (score, all_preds) for full compatibility.
def test(model, test_loader,
         scorer=matthews_corrcoef,
         output_transform=lambda x: (x > 0.5).float()):
    """
    Evaluates the model on a test set.

    Returns:
        tuple: (score, all_preds)  — metric value and list of predictions.
    """
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            preds = output_transform(outputs)
            all_preds.extend(preds.cpu().numpy().flatten())
            all_labels.extend(batch_y.cpu().numpy().flatten())
    score = scorer(all_labels, all_preds)
    return score, all_preds


# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


# 3. Hyperparameter Tuning with Ray Tune

In [4]:
from ray import tune
from ray.tune.schedulers import ASHAScheduler
import random
import ray
import gc

if ray.is_initialized():
    ray.shutdown()

ray.init(object_store_memory=2 * 1024 * 1024 * 1024)


/Users/negin/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-18 23:00:05,478	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-03-18 23:00:05,637	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-03-18 23:00:07,776	INFO worker.py:2013 -- Started a local Ray instance.
/Users/negin/miniconda3/lib/python3.13/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON

Python version:,3.13.1
Ray version:,2.54.0


(autoscaler +31s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.
(autoscaler +31s) Error: No available node types can fulfill resource request {'CPU': 1.0, 'GPU': 1.0}. Add suitable node types to this cluster to resolve this issue.
(autoscaler +1m6s) Error: No available node types can fulfill resource request {'CPU': 1.0, 'GPU': 1.0}. Add suitable node types to this cluster to resolve this issue.
(autoscaler +1m41s) Error: No available node types can fulfill resource request {'CPU': 1.0, 'GPU': 1.0}. Add suitable node types to this cluster to resolve this issue.
(autoscaler +2m16s) Error: No available node types can fulfill resource request {'CPU': 1.0, 'GPU': 1.0}. Add suitable node types to this cluster to resolve this issue.
(autoscaler +2m51s) Error: No available node types can fulfill resource request {'CPU': 1.0, 'GPU': 1.0}. Add suitable node types to this cluster to resolve this issue.
(autoscaler +3m26s) Error: No 

(test_config pid=15837) /var/folders/8v/dft5tkg533s0f0n7c3jl_mwr0000gn/T/ipykernel_15328/354075176.py:65: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)


(test_config pid=15837) Validation score improved to 0.0000
(test_config pid=15837) Epoch [1/100], Loss: 0.8421, Val MCC: 0.0000
(test_config pid=15838) Validation score improved to 0.0000
(test_config pid=15837) Epoch [2/100], Loss: 0.0808, Val MCC: 0.0000 [repeated 2x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(test_config pid=15838) Epoch [2/100], Loss: 0.1632, Val MCC: 0.0000
(test_config pid=15837) Epoch [3/100], Loss: 0.1663, Val MCC: 0.0000
(test_config pid=15838) Epoch [3/100], Loss: 0.4307, Val MCC: 0.0000
(test_config pid=15837) Epoch [4/100], Loss: 0.8180, Val MCC: 0.0000
(test_config pid=15837) Epoch [5/100], Loss: 0.5508, Val MCC: 0.0000
(test_config pid=15838) Epoch [4/100], Loss: 0.0893, Val MCC: 0.0000
(test_config pid=15837) Epoch [6/100], Loss: 0.5061, Val MCC: 0.0000
(test_config pid=

...

(test_config pid=25017) Validation score improved to 0.0000
(test_config pid=25017) Epoch [1/100], Loss: 0.0610, Val MCC: 0.0000
(test_config pid=23028) Epoch [58/100], Loss: 0.0000, Val MCC: 0.9420
(test_config pid=25017) Epoch [2/100], Loss: 0.0580, Val MCC: 0.0000
(test_config pid=23028) Epoch [59/100], Loss: 0.0002, Val MCC: 0.9158
(test_config pid=25017) Epoch [3/100], Loss: 0.3956, Val MCC: 0.0000
(test_config pid=23028) Epoch [60/100], Loss: 0.0000, Val MCC: 0.9082
(test_config pid=23028) Epoch [61/100], Loss: 0.0000, Val MCC: 0.9177 [repeated 2x across cluster]
(test_config pid=25017) Epoch [6/100], Loss: 0.4681, Val MCC: 0.0000 [repeated 2x across cluster]
(test_config pid=25017) Epoch [7/100], Loss: 0.4202, Val MCC: 0.0000 [repeated 2x across cluster]
(test_config pid=23028) Validation score improved to 0.9576
(test_config pid=25017) Epoch [8/100], Loss: 0.6334, Val MCC: 0.0000 [repeated 2x across cluster]
(test_config pid=23028) Epoch [64/100], Loss: 0.0001, Val MCC: 0.9415


(test_config pid=25159) /var/folders/8v/dft5tkg533s0f0n7c3jl_mwr0000gn/T/ipykernel_15328/354075176.py:65: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)


(test_config pid=25159) Validation score improved to 0.0000
(test_config pid=25159) Epoch [1/100], Loss: 0.5936, Val MCC: 0.0000
(test_config pid=25017) Epoch [33/100], Loss: 0.0216, Val MCC: 0.7516
(test_config pid=25159) Epoch [3/100], Loss: 0.4571, Val MCC: 0.0000 [repeated 2x across cluster]
(test_config pid=25159) Epoch [4/100], Loss: 0.1918, Val MCC: 0.0000 [repeated 2x across cluster]
(test_config pid=25017) Epoch [35/100], Loss: 0.0013, Val MCC: 0.7413
(test_config pid=25159) Epoch [5/100], Loss: 0.2157, Val MCC: 0.0000
(test_config pid=25159) Epoch [6/100], Loss: 0.1661, Val MCC: 0.0000
(test_config pid=25017) Epoch [36/100], Loss: 0.0950, Val MCC: 0.6944
(test_config pid=25159) Validation score improved to 0.0707
(test_config pid=25159) Epoch [8/100], Loss: 0.0915, Val MCC: -0.0097 [repeated 2x across cluster]
(test_config pid=25159) Epoch [9/100], Loss: 0.2019, Val MCC: -0.0124 [repeated 2x across cluster]
(test_config pid=25159) Epoch [10/100], Loss: 0.0563, Val MCC: 0.0247

(test_config pid=26055) /var/folders/8v/dft5tkg533s0f0n7c3jl_mwr0000gn/T/ipykernel_15328/354075176.py:65: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)


(test_config pid=25159) Epoch [60/100], Loss: 0.0000, Val MCC: 0.6264
(test_config pid=26055) Validation score improved to 0.0000
(test_config pid=25159) Epoch [61/100], Loss: 0.0000, Val MCC: 0.6731 [repeated 2x across cluster]
(test_config pid=25159) Epoch [62/100], Loss: 0.0377, Val MCC: 0.6614
(test_config pid=26055) Epoch [2/100], Loss: 0.6385, Val MCC: 0.0000
(test_config pid=25159) Epoch [64/100], Loss: 0.0000, Val MCC: 0.6480 [repeated 2x across cluster]
(test_config pid=25159) Epoch [65/100], Loss: 0.0000, Val MCC: 0.6948
(test_config pid=26055) Epoch [3/100], Loss: 1.0921, Val MCC: 0.0000
(test_config pid=25159) Epoch [66/100], Loss: 0.0004, Val MCC: 0.7121
(test_config pid=25159) Epoch [67/100], Loss: 0.0000, Val MCC: 0.6979
(test_config pid=25159) Epoch [68/100], Loss: 0.0000, Val MCC: 0.7185 [repeated 2x across cluster]
(test_config pid=25159) Epoch [69/100], Loss: 0.0000, Val MCC: 0.7917
(test_config pid=26055) Epoch [5/100], Loss: 0.0984, Val MCC: 0.0000
(test_config pid

In [6]:
dataset = pd.read_csv("../2.Data_Preparation/train_bench.tsv", sep="\t")
all_data = create_one_hot_sets(dataset)
print("Moving data to Ray Object Store...")
data_ref = ray.put(all_data)
del all_data
gc.collect()
print("Data stored. Reference created.")


Moving data to Ray Object Store...
Data stored. Reference created.


In [7]:
def generate_hidden_layer_size(config):
    pool = [1024, 512, 256, 128, 64, 32]
    num_layers = min(config["num_layers"], len(pool))
    chosen_dims = random.sample(pool, num_layers)
    if random.choice([True, False]):
        return sorted(chosen_dims, reverse=True)
    else:
        random.shuffle(chosen_dims)
        return chosen_dims


In [8]:
config = {
    "num_layers": tune.choice([2, 3, 4, 5]),
    "hidden_sizes": tune.sample_from(generate_hidden_layer_size),
    "dropout": tune.uniform(0.1, 0.5),
    "lr": tune.loguniform(1e-4, 1e-3),
    "batch_size": tune.choice([10, 20]),
    "num_lstm_layers": tune.choice([1, 2]),
    "lstm_hidden_size": tune.choice([64, 128]),
    "data_ref": data_ref,
}


In [9]:
def test_config(config):
    """Ray Tune trainable: 5-Fold Cross-Validation for one hyperparameter set."""
    all_data = ray.get(config["data_ref"])
    mcc_scores = []
    _device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for i in range(5):
        training_indices = [(i + 1) % 5, (i + 2) % 5, (i + 3) % 5]
        validation_index = (i + 4) % 5
        testing_index = i

        x_train_conc = np.concatenate([all_data[j][0] for j in training_indices])
        y_train_conc = np.concatenate([all_data[j][1] for j in training_indices])
        x_val, y_val = all_data[validation_index]
        x_test, y_test = all_data[testing_index]

        train_loader = DataLoader(SignalDataset(x_train_conc, y_train_conc),
                                  batch_size=config["batch_size"], shuffle=True)
        val_loader   = DataLoader(SignalDataset(x_val, y_val),
                                  batch_size=config["batch_size"])
        test_loader  = DataLoader(SignalDataset(x_test, y_test),
                                  batch_size=config["batch_size"])

        model = SP_NN(
            input_size=21,
            hidden_sizes=config["hidden_sizes"],
            lstm_hidden_size=config["lstm_hidden_size"],
            num_lstm_layers=config["num_lstm_layers"],
            output_size=1,
            dropout_p=config["dropout"]
        ).to(_device)

        optimizer = optim.Adam(model.parameters(), lr=config["lr"])
        criterion = nn.BCELoss()

        best_state = train_val(model, train_loader, val_loader,
                               optimizer, criterion, epochs=100, patience=20)
        model.load_state_dict(best_state)

        # FIX: test() now returns (score, preds); unpack accordingly
        mcc, _ = test(model, test_loader)
        mcc_scores.append(mcc)

    mean_mcc = np.mean(mcc_scores)
    tune.report({"mcc": mean_mcc, "loss": -mean_mcc,
                 "hidden_layers_size": str(config["hidden_sizes"])})


In [11]:
result = tune.run(
    test_config,
    config=config,
    num_samples=15,
    resources_per_trial={"cpu": 4, "gpu": 0}
)

best_trial = result.get_best_trial("mcc", "max", "last")
print("Best trial config:", best_trial.config)
print("Best CV MCC:", best_trial.last_result["mcc"])


2026-03-18 23:37:09,726	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


Trial name,hidden_layers_size,loss,mcc
test_config_0011c_00000,"[1024, 512, 64, 32]",-0.894413,0.894413
test_config_0011c_00001,"[1024, 128, 64, 32]",-0.717879,0.717879
test_config_0011c_00002,"[1024, 64, 32, 256, 128]",-0.781021,0.781021
test_config_0011c_00003,"[128, 32, 64]",-0.726267,0.726267
test_config_0011c_00004,"[1024, 512, 256, 128, 32]",-0.841003,0.841003
test_config_0011c_00005,"[512, 32, 64]",-0.672449,0.672449
test_config_0011c_00006,"[1024, 256, 128, 64]",-0.0478173,0.0478173
test_config_0011c_00007,"[1024, 512, 128, 64, 32]",-0.679197,0.679197
test_config_0011c_00008,"[256, 1024, 128, 32, 64]",-0.856355,0.856355
test_config_0011c_00009,"[512, 32, 256]",-0.728616,0.728616


2026-03-19 08:23:40,064	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/Users/negin/ray_results/test_config_2026-03-18_23-37-09' in 0.0083s.
2026-03-19 08:23:40,069	INFO tune.py:1041 -- Total run time: 31590.34 seconds (31590.31 seconds for the tuning loop).


Best trial config: {'num_layers': 5, 'hidden_sizes': [1024, 256, 512, 128, 32], 'dropout': 0.4845882510426389, 'lr': 0.00036719104008913926, 'batch_size': 10, 'num_lstm_layers': 2, 'lstm_hidden_size': 128, 'data_ref': ObjectRef(00ffffffffffffffffffffffffffffffffffffff0100000001e1f505)}
Best CV MCC: 0.9052120471451695
